<a href="https://colab.research.google.com/github/wetherc/data-2000/blob/sp26/exams/final-prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework Assignment: Feed-Forward Neural Networks for Tabular Data

**Objective:** In this assignment, you will build, train, and evaluate a Feed-Forward Neural Network (Multi-Layer Perceptron) to process tabular data. You will use the **MovieLens 100k** dataset to predict user ratings for movies based on various features.

### Instructions
1. Run the provided starter code to load the dataset.
2. Complete the tasks outlined in the markdown cells below.
3. Ensure your code is well-commented and your plots are clearly labeled.
4. Answer any conceptual questions in a separate markdown block.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf.keras.utils.set_random_seed(42)

In [ ]:
dataset, ds_info = tfds.load(
    'titanic',
    split='train',
    with_info=True,
)

# Display some dataset metadata
print(f"\nNumber of examples: {ds_info.splits['train'].num_examples}")
print(f"Dataset features: {list(ds_info.features.keys())}")

## Task 1: Data Exploration
Before building a model, it is crucial to understand your tabular data.

**Your Task:**
1. Extract a batch of records from the dataset (e.g., using `dataset.take(5)`).
2. Print the features and their corresponding values to understand the structure of the data.
3. Identify the target variable (`user_rating`) and continuous/categorical features you might want to use.

In [ ]:
records = []
for example in dataset.take(1309):
    # Convert each tensor value to a numpy value
    record = {feature_name: feature_value.numpy() for feature_name, feature_value in example.items()}
    records.append(record)

df = pd.DataFrame(records)
df.describe()

## Task 2: Data Preprocessing


### Step 1: Extract Specific Features
First, we isolate the specific features we want our neural network to learn from. We extract `user_zip_code`, `user_gender`, `raw_user_age`, and `movie_genres` as our inputs, and `user_rating` as our target variable. We cast numerical and boolean fields to `tf.float32` for compatibility with TensorFlow.

In [ ]:
def preprocess_data(features):
    inputs = {
        'age': tf.cast(features['age'], tf.float32),
        'parch': tf.cast(features['parch'], tf.float32),
        'pclass': tf.cast(features['pclass'], tf.float32),
        'sex': tf.cast(features['sex'], tf.float32),
        'fare': features['fare']
    }
    target = features['survived']
    return inputs, target

processed_dataset = dataset.map(preprocess_data)

In [ ]:
type(processed_dataset)

In [ ]:
_norm_layers = {}
inputs = ['age','parch','pclass','sex','fare']

for input in inputs:
    _norm_layers[input] = tf.keras.layers.Normalization(axis=None)
    _norm_layers[input].adapt(
        dataset.map(lambda x: tf.cast(x[input], tf.float32)).batch(10000)
    )

In [ ]:
_norm_layers

In [ ]:
# Apply the transformations
def encode_features(inputs, target):
    encoded_inputs = {
        'age': _norm_layers['age'](inputs['age']),
        'parch': _norm_layers['parch'](inputs['parch']),
        'pclass': _norm_layers['pclass'](inputs['pclass']),
        'sex': _norm_layers['sex'](inputs['sex']),
        'fare': _norm_layers['fare'](inputs['fare']),
    }
    return encoded_inputs, target

encoded_dataset = processed_dataset.map(encode_features)

### Step 3: Train/Test Split
To properly evaluate our model, we must test it on data it hasn't seen during training. We shuffle the dataset randomly and split it: 80% for training and 20% for testing.

In [ ]:
num_examples = ds_info.splits['train'].num_examples
train_size = int(0.8 * num_examples)

encoded_dataset = encoded_dataset.shuffle(10000, seed=42)

train_dataset = encoded_dataset.take(train_size)
test_dataset = encoded_dataset.skip(train_size)

In [ ]:
batch_size = 32

train_dataset = train_dataset.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
age_input = tf.keras.Input(
    shape=(1,),
    name='age',
    dtype=tf.float32)
parch_input = tf.keras.Input(
    shape=(1,),
    name='parch',
    dtype=tf.float32)
pclass_input = tf.keras.Input(
    shape=(1,),
    name='pclass',
    dtype=tf.float32)
sex_input = tf.keras.Input(
    shape=(1,),
    name='sex',
    dtype=tf.float32)
fare_input = tf.keras.Input(
    shape=(1,),
    name='fare',
    dtype=tf.float32)

concatenated_features = tf.keras.layers.concatenate([
    age_input,
    parch_input,
    pclass_input,
    sex_input,
    fare_input,
])

# Add Dense hidden layers
x = tf.keras.layers.Dense(64, activation='relu')(concatenated_features)
x = tf.keras.layers.Dense(32, activation='relu')(x)

output_layer = tf.keras.layers.Dense(1, activation='linear', name='survived')(x)

# Define the model
model = tf.keras.Model(
    inputs={
        'age': age_input,
        'pclass': pclass_input,
        'parch': parch_input,
        'sex': sex_input,
        'fare': fare_input,
    },
    outputs=output_layer
)

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy,
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=test_dataset
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss (Accuracy)')
plt.plot(history.history['val_loss'], label='Val Loss (Accuracy)')
plt.title('Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (Accuracy)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()